In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

In [5]:
orders.head()
orders.shape
orders.info()
orders.describe()
orders.isnull().sum()
orders.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


np.int64(0)

In [6]:
df = orders.merge(items, on="order_id", how="left")

df = df.merge(
    customers,
    on="customer_id",
    how="left"
)

df = df.merge(
    sellers,
    on="seller_id",
    how="left",
    suffixes=("_customer", "_seller")
)

df = df.merge(
    products,
    on="product_id",
    how="left"
)

df.shape
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113264 non-null  object 
 5   order_delivered_carrier_date   111457 non-null  object 
 6   order_delivered_customer_date  110196 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   order_item_id                  112650 non-null  float64
 9   product_id                     112650 non-null  object 
 10  seller_id                      112650 non-null  object 
 11  shipping_limit_date            112650 non-null  object 
 12  price                         

In [8]:
import os
import numpy as np
import pandas as pd

def clean_olist_data(df, save_path=None):
    """
    Clean and preprocess the merged Olist logistics dataframe.

    Parameters
    ----------
    df : pandas.DataFrame
        Merged dataframe created in Step 4.

    save_path : str, optional
        Location where the cleaned CSV file will be saved.

    Returns
    -------
    df_clean : pandas.DataFrame
        Cleaned item-level dataframe.

    cleaning_report : pandas.DataFrame
        Summary of rows removed during each cleaning stage.
    """

    # --------------------------------------------------------
    # 1. Create a copy of the original dataframe
    # --------------------------------------------------------

    df_clean = df.copy()

    cleaning_steps = []

    def record_step(step_name, rows_before, rows_after):
        """Store the number of rows removed at each stage."""

        cleaning_steps.append({
            "cleaning_step": step_name,
            "rows_before": rows_before,
            "rows_after": rows_after,
            "rows_removed": rows_before - rows_after
        })

    print("=" * 70)
    print("STEP 6: DATA CLEANING AND PREPROCESSING")
    print("=" * 70)

    print(f"Initial number of rows: {len(df_clean):,}")
    print(f"Initial number of columns: {df_clean.shape[1]:,}")

    # --------------------------------------------------------
    # 2. Check that essential columns exist
    # --------------------------------------------------------

    required_columns = [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "price",
        "freight_value"
    ]

    missing_required_columns = [
        column
        for column in required_columns
        if column not in df_clean.columns
    ]

    if missing_required_columns:
        raise KeyError(
            "The following required columns are missing: "
            + ", ".join(missing_required_columns)
        )

    print("\nAll essential columns are available.")

    # --------------------------------------------------------
    # 3. Standardise text columns
    # --------------------------------------------------------

    rows_before = len(df_clean)

    text_columns = df_clean.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        df_clean[column] = (
            df_clean[column]
            .astype("string")
            .str.strip()
        )

        # Convert empty or invalid text into missing values
        df_clean[column] = df_clean[column].replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "null": pd.NA
        })

    record_step(
        "Standardise text values",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 4. Convert date columns
    # --------------------------------------------------------

    date_columns = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]

    existing_date_columns = [
        column
        for column in date_columns
        if column in df_clean.columns
    ]

    for column in existing_date_columns:
        df_clean[column] = pd.to_datetime(
            df_clean[column],
            errors="coerce"
        )

    print("\nDate columns converted:")
    for column in existing_date_columns:
        print(f"- {column}")

    # --------------------------------------------------------
    # 5. Keep only delivered orders
    # --------------------------------------------------------

    rows_before = len(df_clean)

    df_clean["order_status"] = (
        df_clean["order_status"]
        .str.lower()
    )

    df_clean = df_clean[
        df_clean["order_status"] == "delivered"
    ].copy()

    record_step(
        "Keep delivered orders only",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 6. Remove records without essential dates
    # --------------------------------------------------------

    rows_before = len(df_clean)

    essential_date_columns = [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]

    df_clean = df_clean.dropna(
        subset=essential_date_columns
    ).copy()

    record_step(
        "Remove records with missing essential dates",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 7. Remove records with impossible date sequences
    # --------------------------------------------------------

    rows_before = len(df_clean)

    valid_purchase_and_delivery = (
        df_clean["order_delivered_customer_date"]
        >= df_clean["order_purchase_timestamp"]
    )

    valid_purchase_and_estimate = (
        df_clean["order_estimated_delivery_date"]
        >= df_clean["order_purchase_timestamp"]
    )

    df_clean = df_clean[
        valid_purchase_and_delivery
        & valid_purchase_and_estimate
    ].copy()

    record_step(
        "Remove impossible date sequences",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 8. Create the prediction target safely
    # --------------------------------------------------------
    # 1 = delivered after estimated date
    # 0 = delivered on or before estimated date

    df_clean["late_delivery"] = (
        df_clean["order_delivered_customer_date"]
        > df_clean["order_estimated_delivery_date"]
    ).astype("int8")

    # Actual delay is useful for EDA only.
    # Do not use this column as a model input.

    df_clean["actual_delay_days"] = (
        df_clean["order_delivered_customer_date"]
        - df_clean["order_estimated_delivery_date"]
    ).dt.days

    # --------------------------------------------------------
    # 9. Remove exact duplicate rows
    # --------------------------------------------------------

    rows_before = len(df_clean)

    df_clean = df_clean.drop_duplicates().copy()

    record_step(
        "Remove exact duplicate rows",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 10. Remove duplicate order-item records
    # --------------------------------------------------------
    # Each combination of order_id and order_item_id should
    # normally represent one item within one order.

    if "order_item_id" in df_clean.columns:

        rows_before = len(df_clean)

        duplicated_order_items = df_clean.duplicated(
            subset=["order_id", "order_item_id"],
            keep="first"
        )

        duplicate_count = duplicated_order_items.sum()

        if duplicate_count > 0:
            df_clean = df_clean[
                ~duplicated_order_items
            ].copy()

        record_step(
            "Remove duplicate order-item records",
            rows_before,
            len(df_clean)
        )

    # --------------------------------------------------------
    # 11. Convert numerical columns
    # --------------------------------------------------------

    numerical_columns = [
        "order_item_id",
        "price",
        "freight_value",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "customer_zip_code_prefix",
        "seller_zip_code_prefix"
    ]

    existing_numerical_columns = [
        column
        for column in numerical_columns
        if column in df_clean.columns
    ]

    for column in existing_numerical_columns:
        df_clean[column] = pd.to_numeric(
            df_clean[column],
            errors="coerce"
        )

    # Replace positive and negative infinity with missing values
    df_clean = df_clean.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # --------------------------------------------------------
    # 12. Remove records without price or freight information
    # --------------------------------------------------------

    rows_before = len(df_clean)

    df_clean = df_clean.dropna(
        subset=["price", "freight_value"]
    ).copy()

    record_step(
        "Remove missing price or freight records",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 13. Remove invalid price and freight values
    # --------------------------------------------------------

    rows_before = len(df_clean)

    valid_financial_values = (
        (df_clean["price"] >= 0)
        & (df_clean["freight_value"] >= 0)
    )

    df_clean = df_clean[
        valid_financial_values
    ].copy()

    record_step(
        "Remove negative price or freight values",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 14. Mark invalid product measurements as missing
    # --------------------------------------------------------

    product_measurement_columns = [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]

    existing_measurement_columns = [
        column
        for column in product_measurement_columns
        if column in df_clean.columns
    ]

    for column in existing_measurement_columns:
        # Weight and dimensions should be greater than zero
        df_clean.loc[
            df_clean[column] <= 0,
            column
        ] = np.nan

    # --------------------------------------------------------
    # 15. Clean product category
    # --------------------------------------------------------

    if "product_category_name" in df_clean.columns:
        df_clean["product_category_name"] = (
            df_clean["product_category_name"]
            .fillna("unknown")
            .str.lower()
            .str.strip()
        )

    # --------------------------------------------------------
    # 16. Fill missing product measurements
    # --------------------------------------------------------
    # First use the median of the same product category.
    # If the category median is unavailable, use the global
    # median of the entire column.

    for column in existing_measurement_columns:

        if "product_category_name" in df_clean.columns:

            category_median = (
                df_clean
                .groupby("product_category_name")[column]
                .transform("median")
            )

            df_clean[column] = (
                df_clean[column]
                .fillna(category_median)
            )

        global_median = df_clean[column].median()

        if pd.isna(global_median):
            global_median = 0

        df_clean[column] = (
            df_clean[column]
            .fillna(global_median)
        )

    # --------------------------------------------------------
    # 17. Fill other numerical columns
    # --------------------------------------------------------

    optional_numerical_columns = [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]

    for column in optional_numerical_columns:

        if column in df_clean.columns:
            column_median = df_clean[column].median()

            if pd.isna(column_median):
                column_median = 0

            df_clean[column] = (
                df_clean[column]
                .fillna(column_median)
            )

    # --------------------------------------------------------
    # 18. Clean categorical columns
    # --------------------------------------------------------

    categorical_columns = [
        "product_category_name",
        "customer_city",
        "customer_state",
        "seller_city",
        "seller_state"
    ]

    for column in categorical_columns:

        if column in df_clean.columns:
            df_clean[column] = (
                df_clean[column]
                .fillna("unknown")
                .astype("string")
                .str.strip()
            )

    # State codes should use uppercase letters
    state_columns = [
        "customer_state",
        "seller_state"
    ]

    for column in state_columns:

        if column in df_clean.columns:
            df_clean[column] = (
                df_clean[column]
                .str.upper()
            )

    # City and category names can use lowercase
    lowercase_columns = [
        "customer_city",
        "seller_city",
        "product_category_name"
    ]

    for column in lowercase_columns:

        if column in df_clean.columns:
            df_clean[column] = (
                df_clean[column]
                .str.lower()
            )

    # --------------------------------------------------------
    # 19. Remove records without essential IDs
    # --------------------------------------------------------

    rows_before = len(df_clean)

    essential_id_columns = ["order_id"]

    if "customer_id" in df_clean.columns:
        essential_id_columns.append("customer_id")

    if "product_id" in df_clean.columns:
        essential_id_columns.append("product_id")

    if "seller_id" in df_clean.columns:
        essential_id_columns.append("seller_id")

    df_clean = df_clean.dropna(
        subset=essential_id_columns
    ).copy()

    record_step(
        "Remove records with missing essential IDs",
        rows_before,
        len(df_clean)
    )

    # --------------------------------------------------------
    # 20. Standardise identification columns
    # --------------------------------------------------------

    id_columns = [
        "order_id",
        "customer_id",
        "customer_unique_id",
        "product_id",
        "seller_id"
    ]

    for column in id_columns:

        if column in df_clean.columns:
            df_clean[column] = (
                df_clean[column]
                .astype("string")
                .str.strip()
            )

    # --------------------------------------------------------
    # 21. Reset dataframe index
    # --------------------------------------------------------

    df_clean = df_clean.reset_index(drop=True)

    # --------------------------------------------------------
    # 22. Create the cleaning report
    # --------------------------------------------------------

    cleaning_report = pd.DataFrame(cleaning_steps)

    total_rows_removed = len(df) - len(df_clean)

    print("\n" + "=" * 70)
    print("CLEANING COMPLETED")
    print("=" * 70)

    print(f"Original rows: {len(df):,}")
    print(f"Final rows: {len(df_clean):,}")
    print(f"Total rows removed: {total_rows_removed:,}")
    print(f"Final columns: {df_clean.shape[1]:,}")

    # --------------------------------------------------------
    # 23. Display target distribution
    # --------------------------------------------------------

    print("\nLate-delivery distribution:")

    target_counts = (
        df_clean["late_delivery"]
        .value_counts()
        .sort_index()
    )

    target_percentages = (
        df_clean["late_delivery"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    delivery_distribution = pd.DataFrame({
        "number_of_records": target_counts,
        "percentage": target_percentages
    })

    delivery_distribution.index = [
        "On time" if value == 0 else "Late"
        for value in delivery_distribution.index
    ]

    print(delivery_distribution)

    # --------------------------------------------------------
    # 24. Display remaining missing values
    # --------------------------------------------------------

    remaining_missing = (
        df_clean
        .isnull()
        .sum()
        .sort_values(ascending=False)
    )

    remaining_missing = remaining_missing[
        remaining_missing > 0
    ]

    print("\nColumns still containing missing values:")

    if remaining_missing.empty:
        print("No missing values remain.")
    else:
        print(remaining_missing)

    # --------------------------------------------------------
    # 25. Save the cleaned dataset
    # --------------------------------------------------------

    if save_path is not None:

        output_directory = os.path.dirname(save_path)

        if output_directory:
            os.makedirs(
                output_directory,
                exist_ok=True
            )

        df_clean.to_csv(
            save_path,
            index=False
        )

        print(f"\nCleaned dataset saved to:\n{save_path}")

    return df_clean, cleaning_report

In [9]:
cleaned_file_path = (
    f"E:\DataMining\Logistic\data\processed"
    "cleaned_item_level_data.csv"
)

df_clean, cleaning_report = clean_olist_data(
    df=df,
    save_path=cleaned_file_path
)

STEP 6: DATA CLEANING AND PREPROCESSING
Initial number of rows: 113,425
Initial number of columns: 29

All essential columns are available.

Date columns converted:
- order_purchase_timestamp
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date

CLEANING COMPLETED
Original rows: 113,425
Final rows: 110,189
Total rows removed: 3,236
Final columns: 31

Late-delivery distribution:
         number_of_records  percentage
On time             101475       92.09
Late                  8714        7.91

Columns still containing missing values:
order_approved_at               15
order_delivered_carrier_date     1
dtype: int64

Cleaned dataset saved to:
E:\DataMining\Logistic\data\processedcleaned_item_level_data.csv


In [10]:
display(cleaning_report)

print("Cleaned dataframe shape:", df_clean.shape)

display(df_clean.head())

,cleaning_step,rows_before,rows_after,rows_removed
0,Standardise text values,113425,113425,0
1,Keep delivered orders only,113425,110197,3228
2,Remove records with missing essential dates,110197,110189,8
3,Remove impossible date sequences,110189,110189,0
4,Remove exact duplicate rows,110189,110189,0
5,Remove duplicate order-item records,110189,110189,0
6,Remove missing price or freight records,110189,110189,0
7,Remove negative price or freight values,110189,110189,0
8,Remove records with missing essential IDs,110189,110189,0


Cleaned dataframe shape: (110189, 31)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,late_delivery,actual_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,0,-8
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,0,-6
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,0,-18
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,0,-13
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,0,-10


In [11]:
print("Remaining missing values:")

missing_values = (
    df_clean
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

display(missing_values[missing_values > 0])

Remaining missing values:


order_approved_at               15
order_delivered_carrier_date     1
dtype: int64

In [17]:
# Convert timestamp columns to datetime
date_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

# Time-based features
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_weekday"] = df["order_purchase_timestamp"].dt.dayofweek
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# Estimated delivery duration
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"]
    - df["order_purchase_timestamp"]
).dt.days

# Avoid division by zero
df["freight_price_ratio"] = (
    df["freight_value"]
    / df["price"].replace(0, np.nan)
)

# Product volume
df["product_volume_cm3"] = (
    df["product_length_cm"]
    * df["product_height_cm"]
    * df["product_width_cm"]
)

# 1 means customer and seller are in the same state
df["same_state"] = (
    df["customer_state"] == df["seller_state"]
).astype(int)

# Count the number of items in each order
item_count = (
    items.groupby("order_id")
    .size()
    .reset_index(name="number_of_items")
)

# Prevent duplicate column after rerunning the notebook cell
if "number_of_items" in df.columns:
    df = df.drop(columns=["number_of_items"])

df = df.merge(
    item_count,
    on="order_id",
    how="left"
)

df["number_of_items"] = df["number_of_items"].fillna(0).astype(int)

In [19]:
model_df = df_clean.copy()

order_data = (
    model_df.groupby("order_id", as_index=False)
    .agg({
        "price": "sum",
        "freight_value": "sum",
        "product_weight_g": "sum",
        "product_volume_cm3": "sum",
        "number_of_items": "max",
        "estimated_delivery_days": "first",
        "purchase_month": "first",
        "purchase_weekday": "first",
        "purchase_hour": "first",
        "same_state": "first",
        "customer_state": "first",
        "seller_state": "first",
        "late_delivery": "first"
    })
)

KeyError: "Column(s) ['estimated_delivery_days', 'number_of_items', 'product_volume_cm3', 'purchase_hour', 'purchase_month', 'purchase_weekday', 'same_state'] do not exist"

In [ ]:
order_data.to_csv(
    "../data/processed/prepared_logistics_data.csv",
    index=False
)